In [13]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import time

headers = {
    "User-Agent": "Mozilla/5.0"
}

tags = ["ai", "teknologi", "ekonomi", "kesehatan", "pendidikan"]

data = []
visited = set()

TARGET_TOTAL = 1000
TARGET_PER_TAG = TARGET_TOTAL // len(tags) 

for tag in tags:
    count_per_tag = 0

    print(f"\n{'='*50}")
    print(f"🏷️  Scraping tag: {tag}")
    print(f"{'='*50}")

    
    for page in range(1, 100):  # pagination
        print(f"\n📄 Page {page}")
        
        BASE_URL = f"https://www.kompas.com/tag/{tag}?page={page}"
        
        try:
            res = requests.get(BASE_URL, headers=headers, timeout=10)
            res.raise_for_status()
            soup = BeautifulSoup(res.text, "html.parser")

            # ✅ UPDATE: Gunakan selector yang baru
            articles = soup.find_all("a", class_="article-link")
            print(f"   Found {len(articles)} articles on this page")

            if len(articles) == 0:
                print(f"   ⚠️  No articles found, stopping tag '{tag}'")
                break

            for a in tqdm(articles):
                if count_per_tag >= TARGET_PER_TAG:
                    break

                link = a.get("href")

                if not link or link in visited:
                    continue

                visited.add(link)

                try:
                    art_res = requests.get(link, headers=headers, timeout=10)
                    art_soup = BeautifulSoup(art_res.text, "html.parser")

                    # ✅ UPDATE: Coba cari title dengan berbagai selector
                    title = art_soup.find("h1")
                    if not title:
                        title = art_soup.find("h2")
                    
                    # ✅ UPDATE: Coba berbagai content selector
                    content = art_soup.find("div", class_="read__content")
                    if not content:
                        content = art_soup.find("div", class_="article-content")
                    if not content:
                        # Fallback: cari semua paragraph
                        content = art_soup.find("div", class_="article__body")
                    
                    if not title or not content:
                        continue

                    text = content.get_text(separator=" ").strip()

                    # filter biar gak kosong
                    if len(text.split()) < 50:
                        continue

                    data.append({
                        "title": title.text.strip(),
                        "content": text,
                        "url": link,
                        "tag": tag
                    })

                    count_per_tag += 1

                    print("✅", title.text[:50])

                    time.sleep(0.5)

                except Exception as e:
                    print("ERROR fetching article:", e)
                    continue

            if count_per_tag >= TARGET_PER_TAG:
                break

        except Exception as e:
            print(f"ERROR fetching page: {e}")
            continue

# =========================
# SAVE
# =========================
print(f"\n📊 Total data collected: {len(data)}")

if len(data) > 0:
    df = pd.DataFrame(data)
    df.drop_duplicates(subset="content", inplace=True)
    df.reset_index(drop=True, inplace=True)

    print(f"📊 Total data after dedup: {len(df)}")

    df.to_csv("kompas.csv", index=False)
    print("✅ DONE - File saved as kompas.csv")
else:
    print("❌ NO DATA COLLECTED - Check if the selectors are correct!")



🏷️  Scraping tag: ai

📄 Page 1
   Found 15 articles on this page


  0%|          | 0/15 [00:00<?, ?it/s]

✅ [KLARIFIKASI] Video TKI Histeris Lihat Ledakan Rud


  7%|▋         | 1/15 [00:00<00:12,  1.12it/s]

✅ Geely Percepat Inovasi Berbasis AI: Hadirkan Mobil


...

✅ Reformasi Pendidikan, Prabowo Bidik Perbaikan 300.


 80%|████████  | 12/15 [00:10<00:02,  1.10it/s]

✅ Kado Idul Fitri 2026, Sebanyak 15.038 Guru PAI Lul


 87%|████████▋ | 13/15 [00:11<00:01,  1.10it/s]



📊 Total data collected: 1000
📊 Total data after dedup: 1000
✅ DONE - File saved as kompas.csv


In [9]:
# ===== DIAGNOSTIC CELL - CEK STRUKTUR HTML =====
# Jalankan ini dulu untuk lihat apa struktur HTML yang sebenarnya

import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

test_url = "https://www.kompas.com/tag/ai?page=1"
print(f"🔍 Fetching: {test_url}\n")

res = requests.get(test_url, headers=headers, timeout=10)
print(f"Status Code: {res.status_code}")
print(f"Content Length: {len(res.text)}")

soup = BeautifulSoup(res.text, "html.parser")

# Coba berbagai selector
print("\n" + "="*60)
print("MENCARI ARTICLE LINKS:")
print("="*60)

selectors = [
    ("a", "article__link"),
    ("a", "article-link"),
    ("article", None),
    ("div", "article"),
    ("div", "article-item"),
    ("a", "card"),
]

for tag, class_name in selectors:
    if class_name:
        items = soup.find_all(tag, class_=class_name)
    else:
        items = soup.find_all(tag)
    print(f"\n✓ {tag} with class='{class_name}' : {len(items)} items")
    if items and len(items) > 0:
        print(f"  First item: {str(items[0])[:150]}...")

# Coba ambil semua link
print("\n" + "="*60)
print("SEMUA LINKS DI HALAMAN:")
print("="*60)
all_links = soup.find_all("a", href=True)
kompas_links = [link['href'] for link in all_links if 'kompas.com' in link['href'] and '/article/' in link['href']]
print(f"Total links dengan /article/: {len(kompas_links)}")
if kompas_links:
    print(f"Contoh: {kompas_links[0]}")

# Cek struktur article page
if kompas_links:
    print("\n" + "="*60)
    print("STRUKTUR ARTICLE PAGE:")
    print("="*60)
    art_res = requests.get(kompas_links[0], headers=headers, timeout=10)
    art_soup = BeautifulSoup(art_res.text, "html.parser")
    
    # Cek title
    h1 = art_soup.find("h1")
    print(f"\n✓ <h1>: {'Found' if h1 else 'NOT FOUND'}")
    if h1:
        print(f"  Text: {h1.text[:100]}")
    
    # Cek content dengan berbagai selector
    content_selectors = [
        ("div", "read__content"),
        ("div", "article-content"),
        ("div", "article__content"),
        ("article", None),
        ("div", "content"),
    ]
    
    for tag, class_name in content_selectors:
        if class_name:
            cont = art_soup.find(tag, class_=class_name)
        else:
            cont = art_soup.find(tag)
        status = "✓ Found" if cont else "✗ NOT FOUND"
        print(f"\n{status} - {tag} class='{class_name}'")
        if cont:
            text = cont.get_text(separator=" ").strip()
            print(f"  Word count: {len(text.split())}")
            print(f"  Preview: {text[:100]}...")


🔍 Fetching: https://www.kompas.com/tag/ai?page=1

Status Code: 200
Content Length: 143385

MENCARI ARTICLE LINKS:

✓ a with class='article__link' : 0 items

✓ a with class='article-link' : 15 items
  First item: <a class="article-link" href="https://money.kompas.com/read/2026/04/24/200620326/pasar-ai-indonesia-tumbuh-cepat-kolaborasi-teknologi-menguat">
<div c...

✓ article with class='None' : 0 items

✓ div with class='article' : 0 items

✓ div with class='article-item' : 0 items

✓ a with class='card' : 0 items

SEMUA LINKS DI HALAMAN:
Total links dengan /article/: 0


In [14]:
print(len(df))

1000


In [15]:
df['tag'].value_counts()

tag
ai            200
teknologi     200
ekonomi       200
kesehatan     200
pendidikan    200
Name: count, dtype: int64